In [ ]:
# install required packages
!pip install pandas numpy torch scikit-learn transformers sentencepiece accelerate protobuf

In [ ]:
# imports and setup
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import tokenizers
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, accuracy_score

# constants
TRAINING_SET = './data/train.csv'
VALIDATION_SET = './data/dev.csv'
TEST_SET_INPUT = './data/test.csv'

OUTPUT_FILE = f'Group_57_C.csv'
MODEL_SAVE_PATH = 'Group_57_C_model.pt'

MAX_LEN = 128
MODEL_NAME = "FacebookAI/roberta-base"
BATCH_SIZE = 64
LEARNING_RATE = 1.5e-5
NUM_EPOCHS = 10

def random_seed(seed_value):
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"seeding applied with value: {seed_value}")

random_seed(7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"using device: {device}")

In [ ]:
# data loading n preprocessing
import csv

# grab the csvs and clean them
def prepare_df_features(path):
    try:
        df = pd.read_csv(path)
        # make sure strings and fill nans
        df['premise'] = df['premise'].astype(str).fillna('')
        df['hypothesis'] = df['hypothesis'].astype(str).fillna('')
        return df
    except FileNotFoundError:
        print(f"file not found: {path}")
        return pd.DataFrame()

train_df = prepare_df_features(TRAINING_SET)
val_df = prepare_df_features(VALIDATION_SET)

transformer_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

# custom dataset class to feed the premise-hypothesis pairs into roberta
class nlu_dataset(Dataset):
    def __init__(self, input_df, tokenizer_object, maximum_length, is_test_mode=False):
        self.data = input_df
        self.tokenizer = tokenizer_object
        self.max_len = maximum_length 
        self.is_test_mode = is_test_mode
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        row_data = self.data.iloc[index]
        
        premise = row_data['premise']
        hypothesis = row_data['hypothesis']

        # tokenizes them together with a separator in the middle
        encoded = self.tokenizer(
            premise,
            hypothesis,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation='longest_first', 
            return_tensors='pt'
        )
        
        data_item = {
            'input_ids': encoded['input_ids'].flatten(),
            'attention_mask': encoded['attention_mask'].flatten()
        }
            
        if not self.is_test_mode:
            data_item['labels'] = torch.tensor(row_data['label'], dtype=torch.long)
            
        return data_item

train_dataset = nlu_dataset(train_df, transformer_tokenizer, MAX_LEN)
val_dataset = nlu_dataset(val_df, transformer_tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_cosine_schedule_with_warmup
import torch.nn.functional as F

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2
)
model = model.to(device)

# group the layers so that they can have different learning rates
base_params = [p for n, p in model.named_parameters() if "classifier" not in n]
head_params = [p for n, p in model.named_parameters() if "classifier" in n]

# split learning rates: slow for the base, fast for the new head
optimiser = AdamW([
    {'params': base_params, 'lr': 1.5e-5}, # base
    {'params': head_params, 'lr': 5e-5}    # head
], eps=1e-6, weight_decay=0.01)

total_training_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_training_steps * 0.1)

# warm up so that the pre-trained weights don't get destroyed at the start
scheduler = get_cosine_schedule_with_warmup(
    optimiser, 
    num_warmup_steps=warmup_steps, 
    num_training_steps=total_training_steps
)

# focal loss because normal cross entropy pays too much attention to the easier examples
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce_loss)
        # gamma factor downweights the easy stuff so that it focuses on hard pairs
        focal_loss_value = ((1 - pt) ** self.gamma) * ce_loss
        return torch.mean(focal_loss_value)

# apply class weights to focal loss
labels_array = train_df['label'].values
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels_array), y=labels_array)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

loss_function = FocalLoss(weight=class_weights_tensor, gamma=2.0)

In [ ]:
# training loop
def execute_training_epoch(model_obj, iterator, optimiser_obj, scheduler_obj):
    model_obj.train()
    epoch_loss_acc = 0
    
    for batch_index, data_batch in enumerate(iterator):
        optimiser_obj.zero_grad()
        
        batch_ids = data_batch['input_ids'].to(device)
        batch_mask = data_batch['attention_mask'].to(device)
        batch_labels = data_batch['labels'].to(device)

        # mixed precision to save vram
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            model_outputs = model_obj(input_ids=batch_ids, attention_mask=batch_mask)
            # convert back to float32 because my cuda GPU requires it for some reason
            loss = loss_function(model_outputs.logits.to(torch.float32), batch_labels)
            
        loss.backward()
        # stop gradients from exploding
        nn.utils.clip_grad_norm_(model_obj.parameters(), max_norm=1.0)
        optimiser_obj.step()
        scheduler_obj.step()
        
        epoch_loss_acc += loss.item()
        
        if batch_index % 50 == 0 and batch_index > 0:
            print(f"\tStep {batch_index}/{len(iterator)} | Loss: {loss.item():.4f}")
            
    return epoch_loss_acc / len(iterator)
    
# evaluate performance vs the dev set
def execute_evaluation(model_obj, iterator):
    model_obj.eval()
    prediction_list = []
    true_label_list = []
    epoch_loss_acc = 0
    
    with torch.no_grad():
        for data_batch in iterator:
            batch_ids = data_batch['input_ids'].to(device)
            batch_mask = data_batch['attention_mask'].to(device)
            batch_labels = data_batch['labels'].to(device)
            
            token_type_ids = data_batch.get('token_type_ids', None)
            if token_type_ids is not None: token_type_ids = token_type_ids.to(device)
            
            with torch.amp.autocast('cuda'):
                if token_type_ids is not None:
                    outputs = model_obj(input_ids=batch_ids, attention_mask=batch_mask, token_type_ids=token_type_ids)
                else:
                    outputs = model_obj(input_ids=batch_ids, attention_mask=batch_mask)
                loss = loss_function(outputs.logits, batch_labels)
                
            epoch_loss_acc += loss.item()
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            
            prediction_list.extend(preds)
            true_label_list.extend(batch_labels.cpu().numpy())

    macro_f1 = f1_score(true_label_list, prediction_list, average='macro')
    acc = accuracy_score(true_label_list, prediction_list)
    return epoch_loss_acc / len(iterator), macro_f1, acc

best_val_f1_score = 0
print("BEGIN TRAINING")

# run through the epochs and save the best one
for epoch_idx in range(NUM_EPOCHS):
    print(f"\nEpoch: {epoch_idx + 1:02}")
    train_loss = execute_training_epoch(model, train_loader, optimiser, scheduler)
    val_loss, val_f1, val_acc = execute_evaluation(model, val_loader)
    
    print(f"  Train: {train_loss:.4f}, Val: {val_loss:.4f}, Val F1: {val_f1:.4f}, Val Acc: {val_acc:.4f}")
    if val_f1 > best_val_f1_score:
        best_val_f1_score = val_f1
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"NEW BEST SAVED: {best_val_f1_score:.4f}")

In [ ]:
# predict on test set
def generate_predictions(test_csv_path, model_weights_path, output_filename):
    print(f"loading test data from {test_csv_path}...")
    test_df = prepare_df_features(test_csv_path)
    
    if test_df.empty:
        print("test data empty or missing. skipping.")
        return
        
    test_dataset = nlu_dataset(test_df, transformer_tokenizer, MAX_LEN, is_test_mode=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    print(f"loading best model from {model_weights_path}...")
    inference_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    inference_model.load_state_dict(torch.load(model_weights_path))
    inference_model = inference_model.to(device)
    inference_model.eval()
    
    formatted_results = []
    
    print("running predictions...")
    with torch.no_grad():
        for batch_data in test_loader:
            batch_ids = batch_data['input_ids'].to(device)
            batch_mask = batch_data['attention_mask'].to(device)
            
            token_type_ids = batch_data.get('token_type_ids', None)
            if token_type_ids is not None: token_type_ids = token_type_ids.to(device)
            
            with torch.amp.autocast('cuda'):
                if token_type_ids is not None:
                    outputs = inference_model(input_ids=batch_ids, attention_mask=batch_mask, token_type_ids=token_type_ids)
                else:
                    outputs = inference_model(input_ids=batch_ids, attention_mask=batch_mask)
            
            binary_predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            formatted_results.extend(binary_predictions)
            
    output_df = pd.DataFrame({'prediction': formatted_results})
    output_df.to_csv(output_filename, index=False)
    print(f"done! preds saved to {output_filename}")

generate_predictions(VALIDATION_SET, MODEL_SAVE_PATH, OUTPUT_FILE)

In [ ]:
# from sklearn.metrics import f1_score
# import numpy as np
# import torch

# # sweep for the best cutoff instead of just blindly using 0.5
# def find_optimal_threshold(model_path, data_loader):
#     print("sweeping thresholds...")
#     inference_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
#     inference_model.load_state_dict(torch.load(model_path))
#     inference_model = inference_model.to(device)
#     inference_model.eval()
    
#     all_probs = []
#     all_labels = []
    
#     with torch.no_grad():
#         for batch_data in data_loader:
#             batch_ids = batch_data['input_ids'].to(device)
#             batch_mask = batch_data['attention_mask'].to(device)
#             batch_labels = batch_data['labels']
            
#             outputs = inference_model(input_ids=batch_ids, attention_mask=batch_mask)
            
#             # softmax to get raw probabilities for the class 1
#             probs = torch.nn.functional.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
            
#             all_probs.extend(probs)
#             all_labels.extend(batch_labels.numpy())
            
#     best_threshold = 0.5
#     best_f1_score = 0.0
    
#     # shallow but worthwhile to increase f1
#     possible_thresholds = np.arange(0.3, 0.71, 0.01)
#     for threshold_value in possible_thresholds:
#         binary_preds = (np.array(all_probs) >= threshold_value).astype(int)
#         score = f1_score(all_labels, binary_preds, average='macro')
        
#         if score > best_f1_score:
#             best_f1_score = score
#             best_threshold = threshold_value
            
#     default_f1 = f1_score(all_labels, (np.array(all_probs) >= 0.5).astype(int), average='macro')
#     print(f"default threshold: 0.50 -> old f1: {default_f1:.4f}")
#     print(f"optimal threshold: {best_threshold:.2f} -> new f1: {best_f1_score:.4f}")

# find_optimal_threshold(MODEL_SAVE_PATH, val_loader)